In [ ]:
!pip install pycaret

Pycaret

In [ ]:
# Preparar datos
from pycaret.classification import *

# Recombinar train para PyCaret
train_pycaret = X_train_scaled.copy()
train_pycaret['churn'] = y_train.values

test_pycaret = X_test_scaled.copy()
test_pycaret['churn'] = y_test.values

# Setup de PyCaret
clf = setup(
    data=train_pycaret,
    target='churn',
    session_id=123,
    normalize=False,
    transformation=False,
    fold=5,
    test_data=test_pycaret,
    verbose=False
)

RuntimeError: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')

In [ ]:
# 4. Comparar modelos
# Lista de modelos a incluir (códigos de PyCaret):
# 'svm' - Support Vector Machine
# 'rf' - Random Forest
# 'ada' - AdaBoost
# 'lr' - Logistic Regression
# 'gbc' - Gradient Boosting
# 'xgboost' - XGBoost
# 'lightgbm' - LightGBM
# 'dt' - Decision Tree
# 'knn' - K Neighbors

modelos_incluir = ['svm', 'rf', 'ada', 'lr', 'gbc', 'lightgbm', 'dt', 'knn']

best_models = compare_models(
    include=modelos_incluir,
    sort='AUC',
    n_select=5
)

# 5. Ver resultados
resultados = pull()
print(resultados)

# 6. Tunear el mejor
best = best_models[0]
tuned_best = tune_model(best, optimize='AUC')

# 7. Evaluar en test
predictions = predict_model(tuned_best, data=test_pycaret)

# 8. Ver métricas finales
plot_model(tuned_best, plot='auc')
plot_model(tuned_best, plot='confusion_matrix')

In [ ]:
# MODELO SVM BASELINE
# Crear modelo baseline con class_weight='balanced' para datos desbalanceados
svm_baseline = SVC(kernel='rbf', C=1.0, gamma='scale',
                   class_weight='balanced',  # ← CLAVE PARA DATOS DESBALANCEADOS
                   random_state=42, probability=True)

# Entrenar
svm_baseline.fit(X_train_scaled, y_train)

# Predicciones
y_pred_baseline = svm_baseline.predict(X_test_scaled)
y_pred_proba_baseline = svm_baseline.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Métricas de desempeño
print("\nMÉTRICAS DE DESEMPEÑO:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_baseline):.4f}")
print(f"Recall (Sensitivity): {recall_score(y_test, y_pred_baseline):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_baseline):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_baseline):.4f}")

print("\nREPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, y_pred_baseline,
                          target_names=['No Churn', 'Churn']))


MÉTRICAS DE DESEMPEÑO:
Accuracy: 0.8870
Precision: 0.6086
Recall (Sensitivity): 0.8277
F1-Score: 0.7014
ROC-AUC: 0.9317

REPORTE DE CLASIFICACIÓN:
              precision    recall  f1-score   support

    No Churn       0.96      0.90      0.93      1701
       Churn       0.61      0.83      0.70       325

    accuracy                           0.89      2026
   macro avg       0.79      0.86      0.82      2026
weighted avg       0.91      0.89      0.89      2026



In [ ]:
# OPTIMIZACIÓN CON RANDOMIZEDSEARCHCV
# Definición espacio de búsqueda
param_distributions = {
    'C': loguniform(0.01, 100),
    'gamma': loguniform(0.0001, 1),
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'degree': [2, 3, 4, 5],  # Solo para poly
    'class_weight': ['balanced', None]
}

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    SVC(random_state=42, probability=True),
    param_distributions=param_distributions,
    n_iter=50,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Entrenar
random_search.fit(X_train_scaled, y_train)

# Mejores parámetros
print("\nMEJORES HIPERPARÁMETROS:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nMejor ROC-AUC (CV): {random_search.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits


KeyboardInterrupt: 

In [ ]:
# MODELO OPTIMIZADO - EVALUACIÓN
# Mejor modelo
best_svm = random_search.best_estimator_

# Predicciones
y_pred_best = best_svm.predict(X_test_scaled)
y_pred_proba_best = best_svm.predict_proba(X_test_scaled)[:, 1]

# Métricas
print("\nMÉTRICAS DE DESEMPEÑO:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_best):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_best):.4f}")

print("\nREPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, y_pred_best,
                          target_names=['No Churn', 'Churn']))

In [ ]:
# VISUALIZACIÓN DE RESULTADOS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
axes[0].set_title('Matriz de Confusión - SVM Optimizado', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Valor Real')
axes[0].set_xlabel('Predicción')

# Añadir porcentajes en la matriz
for i in range(2):
    for j in range(2):
        percentage = cm[i, j] / cm[i].sum() * 100
        axes[0].text(j+0.5, i+0.7, f'({percentage:.1f}%)',
                    ha='center', va='center', fontsize=10, color='red')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_best)
auc_score = roc_auc_score(y_test, y_pred_proba_best)

axes[1].plot(fpr, tpr, color='darkorange', lw=2,
            label=f'ROC curve (AUC = {auc_score:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curva ROC - SVM Optimizado', fontsize=14, fontweight='bold')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# COMPARACIÓN BASELINE vs OPTIMIZADO

comparison = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'SVM Baseline': [
        accuracy_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_baseline),
        recall_score(y_test, y_pred_baseline),
        f1_score(y_test, y_pred_baseline),
        roc_auc_score(y_test, y_pred_proba_baseline)
    ],
    'SVM Optimizado': [
        accuracy_score(y_test, y_pred_best),
        precision_score(y_test, y_pred_best),
        recall_score(y_test, y_pred_best),
        f1_score(y_test, y_pred_best),
        roc_auc_score(y_test, y_pred_proba_best)
    ]
})

comparison['Mejora (%)'] = ((comparison['SVM Optimizado'] - comparison['SVM Baseline']) /
                            comparison['SVM Baseline'] * 100).round(2)

print(comparison.round(4))

In [ ]:
# ANÁLISIS DE RENDIMIENTO POR KERNEL

results_df = pd.DataFrame(random_search.cv_results_)
kernel_comparison = results_df.groupby('param_kernel').agg({
    'mean_test_score': ['mean', 'max'],
    'mean_fit_time': 'mean'
}).round(4)

kernel_comparison.columns = ['ROC-AUC Promedio', 'ROC-AUC Máximo', 'Tiempo (s)']
print(kernel_comparison)

# **RandomForest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create a Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)

# Train the model
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the model
print("Random Forest Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))

# Visualize the results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
axes[0].set_title('Matriz de Confusión - Random Forest', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Valor real')
axes[0].set_xlabel('Predicción')

# Add percentages to the confusion matrix
for i in range(2):
    for j in range(2):
        percentage = cm_rf[i, j] / cm_rf[i].sum() * 100
        axes[0].text(j+0.5, i+0.7, f'({percentage:.1f}%)',
                    ha='center', va='center', fontsize=10, color='red')

# ROC Curve
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
auc_score_rf = roc_auc_score(y_test, y_pred_proba_rf)

axes[1].plot(fpr_rf, tpr_rf, color='darkorange', lw=2,
            label=f'ROC curve (AUC = {auc_score_rf:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - Random Forest', fontsize=14, fontweight='bold')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**LIGHT GRADIENT BOOSTING MACHINE (LightGBM)**

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm_model.fit(X_train, y_train)

y_pred_lgbm = lgbm_model.predict(X_test)
y_prob_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

acc_lgbm = accuracy_score(y_test, y_pred_lgbm)
auc_lgbm = roc_auc_score(y_test, y_prob_lgbm)

print("\n🔹 LightGBM Results")
print("Accuracy:", acc_lgbm)
print("AUC:", auc_lgbm)
print(confusion_matrix(y_test, y_pred_lgbm))
print(classification_report(y_test, y_pred_lgbm))


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1302, number of negative: 6799
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000789 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1038
[LightGBM] [Info] Number of data points in the train set: 8101, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.160721 -> initscore=-1.652874
[LightGBM] [Info] Start training from score -1.652874

🔹 LightGBM Results
Accuracy: 0.926949654491609
AUC: 0.9539221272554606
[[1661   40]
 [ 108  217]]
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      1701
           1       0.84      0.67      0.75       325

    accuracy                           0.93      2026
   macro avg       0.89      0.82      0.85     

**GRADIENT BOOSTING CLASSIFIER**



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)
y_prob_gb = gb_model.predict_proba(X_test)[:, 1]

acc_gb = accuracy_score(y_test, y_pred_gb)
auc_gb = roc_auc_score(y_test, y_prob_gb)

print("\n🔹 Gradient Boosting Classifier Results")
print("Accuracy:", acc_gb)
print("AUC:", auc_gb)
print(confusion_matrix(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))


🔹 Gradient Boosting Classifier Results
Accuracy: 0.9264560710760118
AUC: 0.9473793696015917
[[1664   37]
 [ 112  213]]
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      1701
           1       0.85      0.66      0.74       325

    accuracy                           0.93      2026
   macro avg       0.89      0.82      0.85      2026
weighted avg       0.92      0.93      0.92      2026



RED NEURONAL

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import scikeras
from scikeras.wrappers import KerasClassifier
from sklearn.pipeline import Pipeline

In [ ]:
!pip install scikeras

In [ ]:
# Definir la arquitectura y las métricas de optimización de la red neuronal

def create_model(data_entrada):
    model = keras.Sequential([
        layers.Dense(128, activation = "relu", input_dim=data_entrada, name = "hidden-dense-128-layer-1"),
        layers.Dropout(0.3),
        layers.Dense(64, activation = "relu", name = "hidden-dense-64-layer-2"),
        layers.Dropout(0.2),
        layers.Dense(1, activation = "sigmoid", name = "output-layer"),
    ])
    # El método de optimización que voy a utilizar
    adam = tf.keras.optimizers.Adam()
    model.compile(loss = "binary_crossentropy", optimizer = adam, metrics=["accuracy"])
    return model

In [ ]:
# nn_estimators = []
# nn_estimators.append(('standardize', StandardScaler())) # Estandarizar los datos de entrada
# nn_estimators.append(('mlp', KerasClassifier(model = create_model(X_train_scaled.shape[1]), epochs = 30, batch_size = 128, validation_split = 0.2)))

# Clasificador_RN = Pipeline(nn_estimators, verbose = False)
# Clasificador_RN.fit(X_train, y_train) # Ingresar datos de entrenamiento, , epochs=30, batch_size=128
# Clasificador_RN.fit(X_train_scaled, y_train)

# Train the KerasClassifier directly on the scaled data
nn_model = KerasClassifier(model=create_model(X_train_scaled.shape[1]), epochs=30, batch_size=128, validation_split=0.2, verbose=1)

nn_model.fit(X_train_scaled, y_train)

In [ ]:
# Make predictions on the scaled test data
y_pred_nn = nn_model.predict(X_test_scaled)
y_pred_proba_nn = nn_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the model
print("Neural Network Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nn):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_nn):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_nn):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_nn):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nn, target_names=['No Churn', 'Churn']))

In [ ]:
# Visualize the results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm_nn = confusion_matrix(y_test, y_pred_nn)
sns.heatmap(cm_nn, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
axes[0].set_title('Matriz de Confusión - Red Neural', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Valor Real')
axes[0].set_xlabel('Predicción')

# Add percentages to the confusion matrix
for i in range(2):
    for j in range(2):
        percentage = cm_nn[i, j] / cm_nn[i].sum() * 100
        axes[0].text(j+0.5, i+0.7, f'({percentage:.1f}%)',
                    ha='center', va='center', fontsize=10, color='red')

# ROC Curve
fpr_nn, tpr_nn, _ = roc_curve(y_test, y_pred_proba_nn)
auc_score_nn = roc_auc_score(y_test, y_pred_proba_nn)

axes[1].plot(fpr_nn, tpr_nn, color='darkorange', lw=2,
            label=f'ROC curve (AUC = {auc_score_nn:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - Red Neuronal', fontsize=14, fontweight='bold')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()